# Crime Analysis Pipeline

In [1]:
# Import modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

## Ingestion

### Lookup Table


Link for LAD <-> PFA data: __https://ckan.publishing.service.gov.uk/dataset/local-authority-district-to-community-safety-partnership-to-pfa-april-2025-lookup-in-ew/resource/e8cc60d8-f3bb-4a29-ad58-821247d88d95__

Link for LSOA <-> LAD data: __https://ckan.publishing.service.gov.uk/dataset/lsoa-2021-to-electoral-ward-2024-to-lad-2024-best-fit-lookup-in-ew__

In [2]:
# Import LAD <-> PAF raw data
raw_lad_pfr = pd.read_csv('../Data/Raw/lookup/lad-pfa.csv')

raw_lad_pfr.head(5)

,LAD25CD,LAD25NM,CSP25CD,CSP25NM,PFA25CD,PFA25NM,ObjectId
0,E06000058,"Bournemouth, Christchurch and Poole",E22000367,Dorset,E23000039,Dorset,1
1,E06000059,Dorset,E22000367,Dorset,E23000039,Dorset,2
2,E06000060,Buckinghamshire,E22000303,Aylesbury Vale,E23000029,Thames Valley,3
3,E06000060,Buckinghamshire,E22000306,Chiltern,E23000029,Thames Valley,4
4,E06000060,Buckinghamshire,E22000311,South Bucks,E23000029,Thames Valley,5


In [3]:
# Import LSOA <-> LAD data:
raw_lsoa_lad = pd.read_csv('../Data/Raw/lookup/lsoa-lad.csv')

raw_lsoa_lad.head(5)

,LSOA21CD,LSOA21NM,LSOA21NMW,WD24CD,WD24NM,WD24NMW,LAD24CD,LAD24NM,LAD24NMW,ObjectId
0,E01012000,Hartlepool 007E,NaN,E05013038,Burn Valley,NaN,E06000001,Hartlepool,NaN,1
1,E01011964,Hartlepool 007B,NaN,E05013038,Burn Valley,NaN,E06000001,Hartlepool,NaN,2
2,E01011999,Hartlepool 007D,NaN,E05013038,Burn Valley,NaN,E06000001,Hartlepool,NaN,3
3,E01011967,Hartlepool 007C,NaN,E05013038,Burn Valley,NaN,E06000001,Hartlepool,NaN,4
4,E01011951,Hartlepool 007A,NaN,E05013038,Burn Valley,NaN,E06000001,Hartlepool,NaN,5


### Population Data

Link for population data: __https://www.ons.gov.uk/peoplepopulationandcommunity/populationandmigration/populationestimates/datasets/lowersuperoutputareamidyearpopulationestimates__

<div class="alert alert-block alert-warning">
<b>Warning:</b> This section may take some time to import, there are 35,000 rows per year. It usually takes ~1min 30seconds to complete each import.
</div>

In [4]:
# Import Population Data Raw
raw_pop_2022 = pd.read_excel(f'../Data/Raw/population/population.xlsx', sheet_name='Mid-2022 LSOA 2021', skiprows=3, usecols=['LAD 2023 Code', 'LAD 2023 Name', 'LSOA 2021 Code', 'LSOA 2021 Name', 'Total'])

raw_pop_2022.head()

,LAD 2023 Code,LAD 2023 Name,LSOA 2021 Code,LSOA 2021 Name,Total
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,1876
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,1117
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,1260
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,1635
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,1984


In [5]:
raw_pop_2023 = pd.read_excel(f'../Data/Raw/population/population.xlsx', sheet_name='Mid-2023 LSOA 2021', skiprows=3, usecols=['LAD 2023 Code', 'LAD 2023 Name', 'LSOA 2021 Code', 'LSOA 2021 Name', 'Total'])

raw_pop_2023.head()

,LAD 2023 Code,LAD 2023 Name,LSOA 2021 Code,LSOA 2021 Name,Total
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,1925
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,1177
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,1320
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,1670
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,2075


In [6]:
raw_pop_2024 = pd.read_excel(f'../Data/Raw/population/population.xlsx', sheet_name='Mid-2024 LSOA 2021', skiprows=3, usecols=['LAD 2023 Code', 'LAD 2023 Name', 'LSOA 2021 Code', 'LSOA 2021 Name', 'Total'])

raw_pop_2024.head()

,LAD 2023 Code,LAD 2023 Name,LSOA 2021 Code,LSOA 2021 Name,Total
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,1898
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,1247
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,1393
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,1669
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,2303


### Deprivation Data

Link for Deprivation data: __https://www.gov.uk/csv-preview/691ded56d140bbbaa59a2a7d/File_7_IoD2025_All_Ranks_Scores_Deciles_Population_Denominators.csv__

In [7]:
# Import Deprivation Data Raw
raw_depr = pd.read_csv(f'../Data/Raw/deprivation/deprivation.csv')

raw_depr.head()


,LSOA code (2021),LSOA name (2021),Local Authority District code (2024),Local Authority District name (2024),Index of Multiple Deprivation (IMD) Score,Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived),Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs),Income Score (rate),Income Rank (where 1 is most deprived),Income Decile (where 1 is most deprived 10% of LSOAs),...,Indoors Sub-domain Score,Indoors Sub-domain Rank (where 1 is most deprived),Indoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Outdoors Sub-domain Score,Outdoors Sub-domain Rank (where 1 is most deprived),Outdoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Total population: mid 2022,Dependent Children aged 0-15: mid 2022,Older population aged 60 and over: mid 2022,Working age population 18-66 (for use with Employment Deprivation Domain): mid 2022
0,E01000001,City of London 001A,E09000001,City of London,8.742,26525,8,0.013,33730,10,...,1.207,1105,1,1.414,1586,1,1795,149,520,1248
1,E01000002,City of London 001B,E09000001,City of London,4.722,31203,10,0.018,33669,10,...,0.355,9591,3,1.839,592,1,1671,81,387,1324
2,E01000003,City of London 001C,E09000001,City of London,9.250,25913,8,0.107,25167,8,...,0.318,10175,4,1.679,903,1,1896,136,432,1469
3,E01000005,City of London 001E,E09000001,City of London,19.884,14807,5,0.211,14836,5,...,0.012,15502,5,2.065,303,1,1737,177,160,1448
4,E01000006,Barking and Dagenham 016A,E09000002,Barking and Dagenham,25.307,10917,4,0.343,7519,3,...,0.399,8934,3,0.400,9136,3,1837,397,225,1260


### Crime Severity Data

In [8]:
# Import crime severity categorised data set
sev = pd.read_csv(f'../Data/Processed/crime-severity-raw/crime-severity-categorised.csv')

sev.head()

,Crime Index,Offence,Weight,Crime Category
0,"1, 4.1/10/2",Homicide,"7,979",Violence and sexual offences
1,2,Attempted murder,"4,663",Violence and sexual offences
2,4.3,Intentional destruction of viable unborn child,15,Violence and sexual offences
3,4.4,Causing death or serious injury by dangerous d...,"1,092",Violence and sexual offences
4,4.6,Causing death by careless driving when under t...,"1,595",Violence and sexual offences


### Crime Data

In [9]:
## Import one months worth of data
police_region = 'merseyside'
year_month = '2026-03' # Get the most recent data available

mssd = pd.read_csv(f'../Data/Raw/crime-data/{police_region}/{year_month}-{police_region}-street.csv')

mssd.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,NaN,2026-03,Merseyside Police,Merseyside Police,-2.871827,53.489763,On or near Gilescroft Avenue,E01006448,Knowsley 001A,Anti-social behaviour,NaN,NaN
1,ca09c84a698c13c40d825f23eefd27b3ce078c6ef79828...,2026-03,Merseyside Police,Merseyside Police,-2.874541,53.485420,On or near Harleston Road,E01006448,Knowsley 001A,Criminal damage and arson,Unable to prosecute suspect,NaN
2,72c3ba95abd85eb01d07a5443bb313dd6584a19a2052bd...,2026-03,Merseyside Police,Merseyside Police,-2.872892,53.488785,On or near Brook Hey Drive,E01006448,Knowsley 001A,Criminal damage and arson,Investigation complete; no suspect identified,NaN
3,489ca1cd895022d499d58ac9e627952d239292e33c8454...,2026-03,Merseyside Police,Merseyside Police,-2.870190,53.485658,On or near Darmond Road,E01006448,Knowsley 001A,Drugs,Under investigation,NaN
4,f9b6bf6a69cfa25e8e430c6d3403421bda8561db1b5b23...,2026-03,Merseyside Police,Merseyside Police,-2.874261,53.490168,On or near Kenbury Close,E01006448,Knowsley 001A,Other theft,Investigation complete; no suspect identified,NaN


## Cleaning & Validation

### Lookup Table

**LAD <-> PFA conversion database**

In [7]:
# Check data types and overall size
raw_lad_pfr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 332 entries, 0 to 331
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   LAD25CD   332 non-null    object
 1   LAD25NM   332 non-null    object
 2   CSP25CD   332 non-null    object
 3   CSP25NM   332 non-null    object
 4   PFA25CD   332 non-null    object
 5   PFA25NM   332 non-null    object
 6   ObjectId  332 non-null    int64 
dtypes: int64(1), object(6)
memory usage: 18.3+ KB


***
All datatypes are correct.
***

In [8]:
# Check for null values
raw_lad_pfr.isnull().sum()

LAD25CD     0
LAD25NM     0
CSP25CD     0
CSP25NM     0
PFA25CD     0
PFA25NM     0
ObjectId    0
dtype: int64

***
No data is null.
***

In [9]:
# Check for duplicated data
raw_lad_pfr.duplicated().sum()

np.int64(0)

***
No duplicated rows.
***

***
**Remove Unneeded Columns**  
Needed Columns:
- LAD Code
- LAD Name
- PFA Code
- PFA Name
***

In [10]:
lad_pfr = raw_lad_pfr[['LAD25CD', 'LAD25NM', 'PFA25CD', 'PFA25NM']]

# Rename columns to easier names

lad_pfr = lad_pfr.rename(columns={
    'LAD25CD': 'lad_code',
    'LAD25NM': 'lad_name',
    'PFA25CD': 'pfa_code',
    'PFA25NM': 'pfa_name'
})

lad_pfr.head()

,lad_code,lad_name,pfa_code,pfa_name
0,E06000058,"Bournemouth, Christchurch and Poole",E23000039,Dorset
1,E06000059,Dorset,E23000039,Dorset
2,E06000060,Buckinghamshire,E23000029,Thames Valley
3,E06000060,Buckinghamshire,E23000029,Thames Valley
4,E06000060,Buckinghamshire,E23000029,Thames Valley


**LSOA <-> LAD conversion database**

In [11]:
# Check data types and overall size
raw_lsoa_lad.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35672 entries, 0 to 35671
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   LSOA21CD   35672 non-null  object
 1   LSOA21NM   35672 non-null  object
 2   LSOA21NMW  1917 non-null   object
 3   WD24CD     35672 non-null  object
 4   WD24NM     35672 non-null  object
 5   WD24NMW    1917 non-null   object
 6   LAD24CD    35672 non-null  object
 7   LAD24NM    35672 non-null  object
 8   LAD24NMW   1917 non-null   object
 9   ObjectId   35672 non-null  int64 
dtypes: int64(1), object(9)
memory usage: 2.7+ MB


***
All datatypes are correct.
***

In [12]:
# Check for null values
raw_lsoa_lad.isnull().sum()

LSOA21CD         0
LSOA21NM         0
LSOA21NMW    33755
WD24CD           0
WD24NM           0
WD24NMW      33755
LAD24CD          0
LAD24NM          0
LAD24NMW     33755
ObjectId         0
dtype: int64

***
Only useless data is null, can be ignored as we will be dropping to essentials shortly.
***

In [13]:
# Check for duplicated data
raw_lsoa_lad.duplicated().sum()

np.int64(0)

***
No duplicated rows.
***

***
**Remove Unneeded Columns**  
Needed Columns:
- LSOA Code
- LSOA Name
- LAD Code
- LAD Name
***

In [14]:
lsoa_lad = raw_lsoa_lad[['LSOA21CD', 'LSOA21NM', 'LAD24CD', 'LAD24NM']]

# Rename columns to easier names

lsoa_lad = lsoa_lad.rename(columns={
    'LSOA21CD': 'lsoa_code',
    'LSOA21NM': 'lsoa_name',
    'LAD24CD': 'lad_code',
    'LAD24NM': 'lad_name'
})

lsoa_lad.head()

,lsoa_code,lsoa_name,lad_code,lad_name
0,E01012000,Hartlepool 007E,E06000001,Hartlepool
1,E01011964,Hartlepool 007B,E06000001,Hartlepool
2,E01011999,Hartlepool 007D,E06000001,Hartlepool
3,E01011967,Hartlepool 007C,E06000001,Hartlepool
4,E01011951,Hartlepool 007A,E06000001,Hartlepool


### Population

In [15]:
raw_pop_2022.info

<bound method DataFrame.info of       LAD 2023 Code   LAD 2023 Name LSOA 2021 Code       LSOA 2021 Name  Total
0         E06000001      Hartlepool      E01011949      Hartlepool 009A   1876
1         E06000001      Hartlepool      E01011950      Hartlepool 008A   1117
2         E06000001      Hartlepool      E01011951      Hartlepool 007A   1260
3         E06000001      Hartlepool      E01011952      Hartlepool 002A   1635
4         E06000001      Hartlepool      E01011953      Hartlepool 002B   1984
...             ...             ...            ...                  ...    ...
35667     W06000024  Merthyr Tydfil      W01001324  Merthyr Tydfil 003E   1888
35668     W06000024  Merthyr Tydfil      W01001898  Merthyr Tydfil 008F   1452
35669     W06000024  Merthyr Tydfil      W01001959  Merthyr Tydfil 005E   1547
35670     W06000024  Merthyr Tydfil      W01001960  Merthyr Tydfil 005F   1481
35671     W06000024  Merthyr Tydfil      W01001961  Merthyr Tydfil 006G   2321

[35672 rows x 5 col

In [16]:
raw_pop_2023.info

<bound method DataFrame.info of       LAD 2023 Code   LAD 2023 Name LSOA 2021 Code       LSOA 2021 Name  Total
0         E06000001      Hartlepool      E01011949      Hartlepool 009A   1925
1         E06000001      Hartlepool      E01011950      Hartlepool 008A   1177
2         E06000001      Hartlepool      E01011951      Hartlepool 007A   1320
3         E06000001      Hartlepool      E01011952      Hartlepool 002A   1670
4         E06000001      Hartlepool      E01011953      Hartlepool 002B   2075
...             ...             ...            ...                  ...    ...
35667     W06000024  Merthyr Tydfil      W01001324  Merthyr Tydfil 003E   1844
35668     W06000024  Merthyr Tydfil      W01001898  Merthyr Tydfil 008F   1437
35669     W06000024  Merthyr Tydfil      W01001959  Merthyr Tydfil 005E   1555
35670     W06000024  Merthyr Tydfil      W01001960  Merthyr Tydfil 005F   1467
35671     W06000024  Merthyr Tydfil      W01001961  Merthyr Tydfil 006G   2333

[35672 rows x 5 col

In [17]:
raw_pop_2024.info

<bound method DataFrame.info of       LAD 2023 Code   LAD 2023 Name LSOA 2021 Code       LSOA 2021 Name  Total
0         E06000001      Hartlepool      E01011949      Hartlepool 009A   1898
1         E06000001      Hartlepool      E01011950      Hartlepool 008A   1247
2         E06000001      Hartlepool      E01011951      Hartlepool 007A   1393
3         E06000001      Hartlepool      E01011952      Hartlepool 002A   1669
4         E06000001      Hartlepool      E01011953      Hartlepool 002B   2303
...             ...             ...            ...                  ...    ...
35667     W06000024  Merthyr Tydfil      W01001324  Merthyr Tydfil 003E   1848
35668     W06000024  Merthyr Tydfil      W01001898  Merthyr Tydfil 008F   1441
35669     W06000024  Merthyr Tydfil      W01001959  Merthyr Tydfil 005E   1528
35670     W06000024  Merthyr Tydfil      W01001960  Merthyr Tydfil 005F   1453
35671     W06000024  Merthyr Tydfil      W01001961  Merthyr Tydfil 006G   2356

[35672 rows x 5 col

In [18]:
raw_pop_2022.isnull().sum()

LAD 2023 Code     0
LAD 2023 Name     0
LSOA 2021 Code    0
LSOA 2021 Name    0
Total             0
dtype: int64

In [19]:
raw_pop_2023.isnull().sum()

LAD 2023 Code     0
LAD 2023 Name     0
LSOA 2021 Code    0
LSOA 2021 Name    0
Total             0
dtype: int64

In [20]:
raw_pop_2024.isnull().sum()

LAD 2023 Code     0
LAD 2023 Name     0
LSOA 2021 Code    0
LSOA 2021 Name    0
Total             0
dtype: int64

In [21]:
raw_pop_2022.duplicated().sum()

np.int64(0)

In [22]:
raw_pop_2023.duplicated().sum()

np.int64(0)

In [23]:
raw_pop_2024.duplicated().sum()

np.int64(0)

***
All data is incredibly clean: correct datatype, no null values, and no duplicates
***

### Deprivation

In [27]:
# Check data types and overall size
raw_depr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33755 entries, 0 to 33754
Data columns (total 56 columns):
 #   Column                                                                                              Non-Null Count  Dtype  
---  ------                                                                                              --------------  -----  
 0   LSOA code (2021)                                                                                    33755 non-null  object 
 1   LSOA name (2021)                                                                                    33755 non-null  object 
 2   Local Authority District code (2024)                                                                33755 non-null  object 
 3   Local Authority District name (2024)                                                                33755 non-null  object 
 4   Index of Multiple Deprivation (IMD) Score                                                           33755 non-nu

***
**Only need columns that are related to data:**  
Keep:  
Local Authority District code (2024)   
Local Authority District name (2024)   
Index of Multiple Deprivation (IMD) Score  
Income Score (rate)  
Employment Score (rate)  
Education, Skills and Training Score   
Barriers to Housing and Services Score  
  

Drop all others.
***

In [28]:
depr = raw_depr[[
    'Local Authority District code (2024)', 
    'Index of Multiple Deprivation (IMD) Score',
    'Income Score (rate)',
    'Employment Score (rate)',
    'Education, Skills and Training Score',
    'Barriers to Housing and Services Score'
]]

depr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33755 entries, 0 to 33754
Data columns (total 6 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Local Authority District code (2024)       33755 non-null  object 
 1   Index of Multiple Deprivation (IMD) Score  33755 non-null  float64
 2   Income Score (rate)                        33755 non-null  float64
 3   Employment Score (rate)                    33755 non-null  float64
 4   Education, Skills and Training Score       33755 non-null  float64
 5   Barriers to Housing and Services Score     33755 non-null  float64
dtypes: float64(5), object(1)
memory usage: 1.5+ MB


***
Data types are all correct.  
We only need to keep rows relating to the districts that we have crime data on.  
We have the LAD code of each  region, we need a new column that tells us its PFA to tell us which police force has jurisdiction over that LAD.
  
**Conclusion:** Create a PFA Code column, by merging the lad-pfa database. To be done in next section.
***

In [29]:
# Check for null values
depr.isnull().sum()

Local Authority District code (2024)         0
Index of Multiple Deprivation (IMD) Score    0
Income Score (rate)                          0
Employment Score (rate)                      0
Education, Skills and Training Score         0
Barriers to Housing and Services Score       0
dtype: int64

***
No nulls in data.
***

In [30]:
# Check for duplicated data
depr.duplicated().sum()

np.int64(0)

***
No duplicated data
***

In [31]:
## Create a new row that looks at the deprivation region and labels it's police region.

## Remove rows where data for areas outside of this projects scope.

### Crime Severity

In [32]:
# Check data types and overall size
sev.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Crime Index     245 non-null    object
 1   Offence         250 non-null    object
 2   Weight          250 non-null    object
 3   Crime Category  242 non-null    object
dtypes: object(4)
memory usage: 7.9+ KB


***
**Data types should be:**  
Crime Index :   object  
Offense:        object  
Weight:         int64  
Crime Category: object  
  
**Conclusion:** Change weight to an integer
***

In [33]:
# Check for nulls
sev.isnull().sum()

Crime Index       5
Offence           0
Weight            0
Crime Category    8
dtype: int64

***
**Crime Index has null values.**
- Looking at the data, it serves no purpose for the reason we need the datasheet for. It is worth dropping the column.  
**Conclusion**: Drop the column

***

**Crime Category has null values.**
- Crime category cannot be dropped, it is required.  
- Set crime category nulls to 'No Category', such that they are known as null.   
- Looking at the data, these offenses are related to cyber risks and hacking. These will not be found within the crime dataset, as it only logs in person activities.  
**Conclusion**: Set null values to 'No Category'
***

In [34]:
# Check for duplicates
print(sev.duplicated().sum())

0


***
**There are 3 duplicated rows in the dataset**  
No reason to keep the duplicated rows in the dataset, it is repeated data, and will skew averages.  
**Conclusion:** Remove dupllicated rows
***
***

In [35]:
# Change weight to an integer
sev['Weight'] = sev['Weight'].astype(str).str.replace(',', '', regex=False)
sev['Weight'] = pd.to_numeric(sev['Weight'], errors='coerce')

print(f'Weight datatype: {sev['Weight'].dtype}')
print(f'Number of nulls in Weight: {sev['Weight'].isnull().sum()}')
print(f'Sample of Weight:')
display(sev['Weight'].sample(3))

## 

Weight datatype: int64
Number of nulls in Weight: 0
Sample of Weight:


68     3344
30      280
160     155
Name: Weight, dtype: int64

In [36]:
#Drop Crime Index
sev = sev.drop(columns=['Crime Index'])

print(f'Sample of severance weighting:')
display(sev.sample(3))

Sample of severance weighting:


,Offence,Weight,Crime Category
101,Aggravated vehicle taking,48,Vehicle crime
48,Rape of a female child under 16,3872,Violence and sexual offences
200,Insurance related fraud,86,Other crime


In [37]:
# Set null values of Crime Category to 'No Category'
sev['Crime Category'] = sev['Crime Category'].fillna('No Category')

print(sev['Crime Category'].value_counts())

Crime Category
Other crime                     93
Violence and sexual offences    83
Burglary                        16
Criminal damage and arson       12
Public order                     9
Other theft                      8
No Category                      8
Possession of weapons            7
Drugs                            5
Vehicle crime                    4
Robbery                          2
Theft from the person            1
Bicycle Theft                    1
Shoplifting                      1
Name: count, dtype: int64


In [38]:
# Remove duplicated rows
sev = sev.drop_duplicates()
print(sev.duplicated().sum())

0


***
***
Data in crime severity is now clean.

In [39]:
sev.info()

<class 'pandas.core.frame.DataFrame'>
Index: 247 entries, 0 to 249
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Offence         247 non-null    object
 1   Weight          247 non-null    int64 
 2   Crime Category  247 non-null    object
dtypes: int64(1), object(2)
memory usage: 7.7+ KB


In [40]:
sev.isnull().sum()

Offence           0
Weight            0
Crime Category    0
dtype: int64

### Crime Data

In [41]:
# Looking at data
mssd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13356 entries, 0 to 13355
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Crime ID               12058 non-null  object 
 1   Month                  13356 non-null  object 
 2   Reported by            13356 non-null  object 
 3   Falls within           13356 non-null  object 
 4   Longitude              13356 non-null  float64
 5   Latitude               13356 non-null  float64
 6   Location               13356 non-null  object 
 7   LSOA code              13356 non-null  object 
 8   LSOA name              13356 non-null  object 
 9   Crime type             13356 non-null  object 
 10  Last outcome category  12058 non-null  object 
 11  Context                0 non-null      float64
dtypes: float64(3), object(9)
memory usage: 1.2+ MB


In [42]:
mssd.isnull().sum()

Crime ID                  1298
Month                        0
Reported by                  0
Falls within                 0
Longitude                    0
Latitude                     0
Location                     0
LSOA code                    0
LSOA name                    0
Crime type                   0
Last outcome category     1298
Context                  13356
dtype: int64

Crime ID has null values - This means we need to create a new primary key for this database.
As each database is categorised by its location and its year and month, we will use that in its primary key.
eg: mers_2026_03_00001 - This allows for up to 100,000 crimes per month per police region.



## Feature Engineering & Transformation

#### Lookup Table

***
This section will merge the lsoa-lad and lad-pfa datasets to create an aggregated lookup table, where each location can be found.
***

In [24]:
# merge on lad code
# final table columns: LSOA Code | LSOA Name | LAD Code | LAD Name | PFA Code | PFA Name

lookup_table = pd.merge(lsoa_lad, lad_pfr, how='outer', on=['lad_code'], indicator=True)

lookup_table.head(5)

,lsoa_code,lsoa_name,lad_code,lad_name_x,lad_name_y,pfa_code,pfa_name,_merge
0,E01012000,Hartlepool 007E,E06000001,Hartlepool,Hartlepool,E23000013,Cleveland,both
1,E01011964,Hartlepool 007B,E06000001,Hartlepool,Hartlepool,E23000013,Cleveland,both
2,E01011999,Hartlepool 007D,E06000001,Hartlepool,Hartlepool,E23000013,Cleveland,both
3,E01011967,Hartlepool 007C,E06000001,Hartlepool,Hartlepool,E23000013,Cleveland,both
4,E01011951,Hartlepool 007A,E06000001,Hartlepool,Hartlepool,E23000013,Cleveland,both


In [25]:
lookup_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38792 entries, 0 to 38791
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   lsoa_code   38790 non-null  object  
 1   lsoa_name   38790 non-null  object  
 2   lad_code    38792 non-null  object  
 3   lad_name_x  38790 non-null  object  
 4   lad_name_y  38301 non-null  object  
 5   pfa_code    38301 non-null  object  
 6   pfa_name    38301 non-null  object  
 7   _merge      38792 non-null  category
dtypes: category(1), object(7)
memory usage: 2.1+ MB


In [26]:
lookup_table.isnull().sum()

lsoa_code       2
lsoa_name       2
lad_code        0
lad_name_x      2
lad_name_y    491
pfa_code      491
pfa_name      491
_merge          0
dtype: int64

In [27]:
lookup_table[lookup_table['lsoa_code'].isna()]

,lsoa_code,lsoa_name,lad_code,lad_name_x,lad_name_y,pfa_code,pfa_name,_merge
31879,NaN,NaN,E08000038,NaN,Barnsley,E23000011,South Yorkshire,right_only
31880,NaN,NaN,E08000039,NaN,Sheffield,E23000011,South Yorkshire,right_only


In [28]:
lookup_table[lookup_table['pfa_code'].isna()]['lad_code'].value_counts()

lad_code
E08000019    343
E08000016    148
Name: count, dtype: int64

***
Two missing values present in the lsoa <-> lad database - for Barnsley and Sheffield in South Yorkshire.    
Furthermore, there are two missing entries in the lad <-> pfa database - again for Barnsley and Sheffield in South Yorkshire. (E...016-E...019)
  
This poses an issue, as these areas are within the crime data this project is looking at.  
  
After doing some reseaching online, the two sectors we are getting issues in were both updated in 2025, according to these links:  
Barnsley: __https://www.ons.gov.uk/explore-local-statistics/areas/E08000038-barnsley__
Sheffield: __https://www.ons.gov.uk/explore-local-statistics/areas/E08000039-sheffield__

This means that these are the same issues, and can be merged to be the same data.

I need to ensure that whenever I am using LAD data before 2025, I update the LAD to E08000039 from the outdated E08000019 and similarly I update to E08000038 from the outdated E08000016.

**Conclusion:** In the lsoa->lad dataset, set any instances of E08000019 or E08000016 to their updated counterparts. Then the datasets can be remerged to create a clean lookup table
***

In [29]:
# Change lad->pfr dataset outdated data

# Update old Sheffield LAD code to new Sheffield LAD code
lsoa_lad.loc[lsoa_lad['lad_code'] == 'E08000016', 'lad_code'] = 'E08000038'

# Update old Barnsley LAD code to new Barnsley LAD code
lsoa_lad.loc[lsoa_lad['lad_code'] == 'E08000019', 'lad_code'] = 'E08000039'

lsoa_lad[lsoa_lad['lad_code'].isin(['E08000038', 'E08000039'])]

# Updated succesfully!

,lsoa_code,lsoa_name,lad_code,lad_name
24051,E01007429,Barnsley 024C,E08000038,Barnsley
24054,E01007444,Barnsley 012F,E08000038,Barnsley
24058,E01007428,Barnsley 024B,E08000038,Barnsley
24059,E01007334,Barnsley 009A,E08000038,Barnsley
24061,E01007382,Barnsley 019A,E08000038,Barnsley
...,...,...,...,...
25184,E01008134,Sheffield 005B,E08000039,Sheffield
25187,E01007888,Sheffield 003A,E08000039,Sheffield
25190,E01007899,Sheffield 003E,E08000039,Sheffield
25193,E01007901,Sheffield 003G,E08000039,Sheffield


***
**Now actually merging the datasets together!**

We made it!

In [30]:
# merge on lad code
# final table columns: LSOA Code | LSOA Name | LAD Code | LAD Name | PFA Code | PFA Name

lookup_table = pd.merge(lsoa_lad, lad_pfr, how='left', on=['lad_code', 'lad_name'])

lookup_table.head(5)

,lsoa_code,lsoa_name,lad_code,lad_name,pfa_code,pfa_name
0,E01012000,Hartlepool 007E,E06000001,Hartlepool,E23000013,Cleveland
1,E01011964,Hartlepool 007B,E06000001,Hartlepool,E23000013,Cleveland
2,E01011999,Hartlepool 007D,E06000001,Hartlepool,E23000013,Cleveland
3,E01011967,Hartlepool 007C,E06000001,Hartlepool,E23000013,Cleveland
4,E01011951,Hartlepool 007A,E06000001,Hartlepool,E23000013,Cleveland


In [31]:
lookup_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38790 entries, 0 to 38789
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   lsoa_code  38790 non-null  object
 1   lsoa_name  38790 non-null  object
 2   lad_code   38790 non-null  object
 3   lad_name   38790 non-null  object
 4   pfa_code   38790 non-null  object
 5   pfa_name   38790 non-null  object
dtypes: object(6)
memory usage: 1.8+ MB


***
All values are the correct datatypes
***

In [32]:
lookup_table.isnull().sum()

lsoa_code    0
lsoa_name    0
lad_code     0
lad_name     0
pfa_code     0
pfa_name     0
dtype: int64

***
No null data
***

In [33]:
lookup_table.duplicated().sum()

np.int64(3118)

***
3118 duplicated rows, drop.
***

In [34]:
# Drop duplicate rows
print(f'row count before dropping: {lookup_table.shape[0]}')

lookup_table = lookup_table.drop_duplicates()
print(f'number of duplicates after dropping: {lookup_table.duplicated().sum()}')

print(f'row count after dropping: {lookup_table.shape[0]}')

row count before dropping: 38790
number of duplicates after dropping: 0
row count after dropping: 35672


***
Now we have a clean lookup database, we can export it to be used later.
***

In [35]:
## Export the code to processed folder as a csv
lookup_table.to_csv('../Data/Processed/lookup-table.csv', index=False)
# Keep the lad_to pfr for databases holding only lad, as to avoid confusion
lad_pfr.to_csv('../Data/Processed/lad-pfr.csv', index=False)

print(f'lookup-table.csv File successfully created: {Path('../Data/Processed/lookup-table.csv').exists()}')
print(f'lad-pfr.csv File successfully created: {Path('../Data/Processed/lad-pfr.csv').exists()}')

lookup-table.csv File successfully created: True
lad-pfr.csv File successfully created: True


#### Population

***
This section will create estimated totals for the populations of LSOA's in 2025 and 2026, then produce a processed database from the now 5 population databases. It will have the columns:  
LSOA Code | LSOA Name | LAD Code | LAD Name | PFR Code | PFR Name |  Total Population_2022/3/4/5/6 (as separate columns)
  
In order to achieve this, the following columns will need to be appended, using the lookup table and other resources:
- PFR Code
- PFR Name

Futhermore, the columns will have their names changed to be easier and standardised. The data definitions and types will be held in the data dictionary.


***

In [36]:
raw_pop_2022.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35672 entries, 0 to 35671
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   LAD 2023 Code   35672 non-null  object
 1   LAD 2023 Name   35672 non-null  object
 2   LSOA 2021 Code  35672 non-null  object
 3   LSOA 2021 Name  35672 non-null  object
 4   Total           35672 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.4+ MB


In [37]:
raw_pop_2023.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35672 entries, 0 to 35671
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   LAD 2023 Code   35672 non-null  object
 1   LAD 2023 Name   35672 non-null  object
 2   LSOA 2021 Code  35672 non-null  object
 3   LSOA 2021 Name  35672 non-null  object
 4   Total           35672 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.4+ MB


In [38]:
raw_pop_2024.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35672 entries, 0 to 35671
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   LAD 2023 Code   35672 non-null  object
 1   LAD 2023 Name   35672 non-null  object
 2   LSOA 2021 Code  35672 non-null  object
 3   LSOA 2021 Name  35672 non-null  object
 4   Total           35672 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.4+ MB


In [47]:
# Rename current columns
raw_pop_2022 = raw_pop_2022.rename(columns={
    'LAD 2023 Code': 'lad_code',
    'LAD 2023 Name': 'lad_name',
    'LSOA 2021 Code': 'lsoa_code',
    'LSOA 2021 Name': 'lsoa_name',
    'Total': 'population',
})

raw_pop_2023 = raw_pop_2023.rename(columns={
    'LAD 2023 Code': 'lad_code',
    'LAD 2023 Name': 'lad_name',
    'LSOA 2021 Code': 'lsoa_code',
    'LSOA 2021 Name': 'lsoa_name',
    'Total': 'population',
})

raw_pop_2024 = raw_pop_2024.rename(columns={
    'LAD 2023 Code': 'lad_code',
    'LAD 2023 Name': 'lad_name',
    'LSOA 2021 Code': 'lsoa_code',
    'LSOA 2021 Name': 'lsoa_name',
    'Total': 'population',
})

raw_pop_2022.head()

,lad_code,lad_name,lsoa_code,lsoa_name,population
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,1876
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,1117
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,1260
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,1635
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,1984


In [42]:
raw_pop_2023.head()

,lad_code,lad_name,lsoa_code,lsoa_name,population
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,1925
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,1177
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,1320
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,1670
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,2075


In [43]:
raw_pop_2024.head()

,lad_code,lad_name,lsoa_code,lsoa_name,population
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,1898
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,1247
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,1393
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,1669
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,2303


***
Fix Sheffield and Barnsley LAD Codes, as data is from before 2025
***

In [48]:
# As data comes from before 2025, we need to update the lad_codes for barnsley and sheffield

# Update old Sheffield LAD code to new Sheffield LAD code
raw_pop_2022.loc[raw_pop_2022['lad_code'] == 'E08000016', 'lad_code'] = 'E08000038'

# Update old Barnsley LAD code to new Barnsley LAD code
raw_pop_2022.loc[raw_pop_2022['lad_code'] == 'E08000019', 'lad_code'] = 'E08000039'

raw_pop_2022[raw_pop_2022['lad_code'].isin(['E08000038', 'E08000039'])]

,lad_code,lad_name,lsoa_code,lsoa_name,population
24051,E08000038,Barnsley,E01007317,Barnsley 018A,1532
24052,E08000038,Barnsley,E01007318,Barnsley 018B,1577
24053,E08000038,Barnsley,E01007319,Barnsley 015A,1423
24054,E08000038,Barnsley,E01007320,Barnsley 018C,1741
24055,E08000038,Barnsley,E01007321,Barnsley 015B,1434
...,...,...,...,...,...
24906,E08000039,Sheffield,E01034842,Sheffield 022I,3476
24907,E08000039,Sheffield,E01034843,Sheffield 036F,3013
24908,E08000039,Sheffield,E01034844,Sheffield 074F,2005
24909,E08000039,Sheffield,E01034845,Sheffield 076G,1389


In [58]:
# As data comes from before 2025, we need to update the lad_codes for barnsley and sheffield

# Update old Sheffield LAD code to new Sheffield LAD code
raw_pop_2023.loc[raw_pop_2023['lad_code'] == 'E08000016', 'lad_code'] = 'E08000038'

# Update old Barnsley LAD code to new Barnsley LAD code
raw_pop_2023.loc[raw_pop_2023['lad_code'] == 'E08000019', 'lad_code'] = 'E08000039'

raw_pop_2023[raw_pop_2023['lad_code'].isin(['E08000038', 'E08000039'])]

,lad_code,lad_name,lsoa_code,lsoa_name,population
24051,E08000038,Barnsley,E01007317,Barnsley 018A,1532
24052,E08000038,Barnsley,E01007318,Barnsley 018B,1577
24053,E08000038,Barnsley,E01007319,Barnsley 015A,1423
24054,E08000038,Barnsley,E01007320,Barnsley 018C,1741
24055,E08000038,Barnsley,E01007321,Barnsley 015B,1434
...,...,...,...,...,...
24906,E08000039,Sheffield,E01034842,Sheffield 022I,3476
24907,E08000039,Sheffield,E01034843,Sheffield 036F,3013
24908,E08000039,Sheffield,E01034844,Sheffield 074F,2005
24909,E08000039,Sheffield,E01034845,Sheffield 076G,1389


In [45]:
# As data comes from before 2025, we need to update the lad_codes for barnsley and sheffield

# Update old Sheffield LAD code to new Sheffield LAD code
raw_pop_2024.loc[raw_pop_2024['lad_code'] == 'E08000016', 'lad_code'] = 'E08000038'

# Update old Barnsley LAD code to new Barnsley LAD code
raw_pop_2024.loc[raw_pop_2024['lad_code'] == 'E08000019', 'lad_code'] = 'E08000039'

raw_pop_2024[raw_pop_2024['lad_code'].isin(['E08000038', 'E08000039'])]

,lad_code,lad_name,lsoa_code,lsoa_name,population
24051,E08000038,Barnsley,E01007317,Barnsley 018A,1540
24052,E08000038,Barnsley,E01007318,Barnsley 018B,1584
24053,E08000038,Barnsley,E01007319,Barnsley 015A,1453
24054,E08000038,Barnsley,E01007320,Barnsley 018C,1785
24055,E08000038,Barnsley,E01007321,Barnsley 015B,1438
...,...,...,...,...,...
24906,E08000039,Sheffield,E01034842,Sheffield 022I,3597
24907,E08000039,Sheffield,E01034843,Sheffield 036F,3118
24908,E08000039,Sheffield,E01034844,Sheffield 074F,2185
24909,E08000039,Sheffield,E01034845,Sheffield 076G,1457


***
The data is now fully prepared to be concatinated. We will now create the 2025 and 2026 predicted populations
***

In [49]:
location_cols = ['lsoa_code', 'lsoa_name', 'lad_code', 'lad_name']

pop_growth = raw_pop_2022[location_cols + ['population']].merge(
    raw_pop_2023[location_cols + ['population']],
    on=location_cols,
    how='inner',
    suffixes=('_2022', '_2023')
).merge(
    raw_pop_2024[location_cols + ['population']],
    on=location_cols,
    how='inner'
)

pop_growth = pop_growth.rename(columns={
    'population': 'population_2024'
})

pop_growth.head()

,lsoa_code,lsoa_name,lad_code,lad_name,population_2022,population_2023,population_2024
0,E01011949,Hartlepool 009A,E06000001,Hartlepool,1876,1925,1898
1,E01011950,Hartlepool 008A,E06000001,Hartlepool,1117,1177,1247
2,E01011951,Hartlepool 007A,E06000001,Hartlepool,1260,1320,1393
3,E01011952,Hartlepool 002A,E06000001,Hartlepool,1635,1670,1669
4,E01011953,Hartlepool 002B,E06000001,Hartlepool,1984,2075,2303


In [50]:
# Calculate average annual change
pop_growth['change_2022_2023'] = (
    pop_growth['population_2023'] - pop_growth['population_2022']
)

pop_growth['change_2023_2024'] = (
    pop_growth['population_2024'] - pop_growth['population_2023']
)

pop_growth['avg_annual_change'] = (
    pop_growth[['change_2022_2023', 'change_2023_2024']].mean(axis=1)
)

pop_growth.head()

,lsoa_code,lsoa_name,lad_code,lad_name,population_2022,population_2023,population_2024,change_2022_2023,change_2023_2024,avg_annual_change
0,E01011949,Hartlepool 009A,E06000001,Hartlepool,1876,1925,1898,49,-27,11.0
1,E01011950,Hartlepool 008A,E06000001,Hartlepool,1117,1177,1247,60,70,65.0
2,E01011951,Hartlepool 007A,E06000001,Hartlepool,1260,1320,1393,60,73,66.5
3,E01011952,Hartlepool 002A,E06000001,Hartlepool,1635,1670,1669,35,-1,17.0
4,E01011953,Hartlepool 002B,E06000001,Hartlepool,1984,2075,2303,91,228,159.5


In [52]:
# Add new column for 2025 estimated population
# ASSUMPTION: Growth rate will stay, on average, the same for the next 2 years after the data

pop_growth['population_2025'] = pop_growth['population_2024'] + pop_growth['avg_annual_change']

pop_growth.head()

,lsoa_code,lsoa_name,lad_code,lad_name,population_2022,population_2023,population_2024,change_2022_2023,change_2023_2024,avg_annual_change,population_2025
0,E01011949,Hartlepool 009A,E06000001,Hartlepool,1876,1925,1898,49,-27,11.0,1909.0
1,E01011950,Hartlepool 008A,E06000001,Hartlepool,1117,1177,1247,60,70,65.0,1312.0
2,E01011951,Hartlepool 007A,E06000001,Hartlepool,1260,1320,1393,60,73,66.5,1459.5
3,E01011952,Hartlepool 002A,E06000001,Hartlepool,1635,1670,1669,35,-1,17.0,1686.0
4,E01011953,Hartlepool 002B,E06000001,Hartlepool,1984,2075,2303,91,228,159.5,2462.5


In [53]:
# Add a new columns for 2026 estimated population

pop_growth['population_2026'] = pop_growth['population_2025'] + pop_growth['avg_annual_change']

pop_growth.head()

,lsoa_code,lsoa_name,lad_code,lad_name,population_2022,population_2023,population_2024,change_2022_2023,change_2023_2024,avg_annual_change,population_2025,population_2026
0,E01011949,Hartlepool 009A,E06000001,Hartlepool,1876,1925,1898,49,-27,11.0,1909.0,1920.0
1,E01011950,Hartlepool 008A,E06000001,Hartlepool,1117,1177,1247,60,70,65.0,1312.0,1377.0
2,E01011951,Hartlepool 007A,E06000001,Hartlepool,1260,1320,1393,60,73,66.5,1459.5,1526.0
3,E01011952,Hartlepool 002A,E06000001,Hartlepool,1635,1670,1669,35,-1,17.0,1686.0,1703.0
4,E01011953,Hartlepool 002B,E06000001,Hartlepool,1984,2075,2303,91,228,159.5,2462.5,2622.0


***
There is now columns for the expected population of each LSOA for 2022-26.  
We will now merge with the lookup table in order to get the PFA code and name for each LSOA as well.  
  
Finally, we will then cut down the table to essential columns, and export it to be used in aggregation.  
***

In [55]:
# Merge with lookup table to add PFA stats

pop_growth_pfa = pd.merge(pop_growth,lookup_table,how='left',on=['lsoa_code', 'lsoa_name', 'lad_code', 'lad_name'])

pop_growth_pfa.head()

,lsoa_code,lsoa_name,lad_code,lad_name,population_2022,population_2023,population_2024,change_2022_2023,change_2023_2024,avg_annual_change,population_2025,population_2026,pfa_code,pfa_name
0,E01011949,Hartlepool 009A,E06000001,Hartlepool,1876,1925,1898,49,-27,11.0,1909.0,1920.0,E23000013,Cleveland
1,E01011950,Hartlepool 008A,E06000001,Hartlepool,1117,1177,1247,60,70,65.0,1312.0,1377.0,E23000013,Cleveland
2,E01011951,Hartlepool 007A,E06000001,Hartlepool,1260,1320,1393,60,73,66.5,1459.5,1526.0,E23000013,Cleveland
3,E01011952,Hartlepool 002A,E06000001,Hartlepool,1635,1670,1669,35,-1,17.0,1686.0,1703.0,E23000013,Cleveland
4,E01011953,Hartlepool 002B,E06000001,Hartlepool,1984,2075,2303,91,228,159.5,2462.5,2622.0,E23000013,Cleveland


In [57]:
## Create final database for population data

population_final = pop_growth_pfa[[
    'lsoa_code', 
    'lsoa_name', 
    'lad_code', 
    'lad_name', 
    'pfa_code', 
    'pfa_name', 
    'population_2022', 
    'population_2023', 
    'population_2024', 
    'population_2025', 
    'population_2026'
]]

population_final.head()

,lsoa_code,lsoa_name,lad_code,lad_name,pfa_code,pfa_name,population_2022,population_2023,population_2024,population_2025,population_2026
0,E01011949,Hartlepool 009A,E06000001,Hartlepool,E23000013,Cleveland,1876,1925,1898,1909.0,1920.0
1,E01011950,Hartlepool 008A,E06000001,Hartlepool,E23000013,Cleveland,1117,1177,1247,1312.0,1377.0
2,E01011951,Hartlepool 007A,E06000001,Hartlepool,E23000013,Cleveland,1260,1320,1393,1459.5,1526.0
3,E01011952,Hartlepool 002A,E06000001,Hartlepool,E23000013,Cleveland,1635,1670,1669,1686.0,1703.0
4,E01011953,Hartlepool 002B,E06000001,Hartlepool,E23000013,Cleveland,1984,2075,2303,2462.5,2622.0


***
Final cleanliness check
***

In [58]:
population_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35181 entries, 0 to 35180
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   lsoa_code        35181 non-null  object 
 1   lsoa_name        35181 non-null  object 
 2   lad_code         35181 non-null  object 
 3   lad_name         35181 non-null  object 
 4   pfa_code         34628 non-null  object 
 5   pfa_name         34628 non-null  object 
 6   population_2022  35181 non-null  int64  
 7   population_2023  35181 non-null  int64  
 8   population_2024  35181 non-null  int64  
 9   population_2025  35181 non-null  float64
 10  population_2026  35181 non-null  float64
dtypes: float64(2), int64(3), object(6)
memory usage: 3.0+ MB


***
population_2025 and population_2026 should be integers, they are denoting populations, and you can't have half a person.
***

In [63]:
population_final.loc[:, 'population_2025'] = (population_final['population_2025'].round().astype(int))

population_final.loc[:, 'population_2026'] = (population_final['population_2026'].round().astype(int))

In [64]:
population_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35181 entries, 0 to 35180
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   lsoa_code        35181 non-null  object
 1   lsoa_name        35181 non-null  object
 2   lad_code         35181 non-null  object
 3   lad_name         35181 non-null  object
 4   pfa_code         34628 non-null  object
 5   pfa_name         34628 non-null  object
 6   population_2022  35181 non-null  int64 
 7   population_2023  35181 non-null  int64 
 8   population_2024  35181 non-null  int64 
 9   population_2025  35181 non-null  int64 
 10  population_2026  35181 non-null  int64 
dtypes: int64(5), object(6)
memory usage: 3.0+ MB


In [69]:
# Check for nulls
population_final.isnull().sum()

lsoa_code            0
lsoa_name            0
lad_code             0
lad_name             0
pfa_code           553
pfa_name           553
population_2022      0
population_2023      0
population_2024      0
population_2025      0
population_2026      0
dtype: int64

In [71]:
# Check for nulls
population_final[population_final.isnull().any(axis=1)]['lad_name'].value_counts()

lad_name
Bristol               268
Kingston upon Hull    168
Herefordshire         117
Name: count, dtype: int64

There are missing PFA's for some areas in Bristol, Kingston and Herefordshire.

Luckily, these are not within the data range that this project is analysing, but should be monitored for the case it is expanded.

**Conclusion:** Drop the missing PFA populations.

In [73]:
print(f'number of rows before dropping missing PFAs:{population_final.shape[0]}')

population_final = population_final.dropna()

print(f'number of rows after dropping missing PFAs:{population_final.shape[0]}')

number of rows before dropping missing PFAs:35181
number of rows after dropping missing PFAs:34628


In [76]:
# Check for duplicate data
population_final.duplicated().sum()

np.int64(0)

***
**Population Data is now officially clean!**  
  
Now export data to be used in final aggregation.
***

In [77]:
## Export the code to processed folder as a csv
population_final.to_csv('../Data/Processed/population.csv', index=False)

print(f'lookup-table.csv File successfully created: {Path('../Data/Processed/population.csv').exists()}')

lookup-table.csv File successfully created: True


#### Deprivation

***
This section will create a processed database from the deprivation database. It will have the columns:  
LAD Code | LAD Name | PFR code | PFR Name | Deprivation Statistics  
  
In order to achieve this, the following columns must be appended by making use of the lookup table:
- LAD Name
- PFR Code
- PFR Name

Furthermore, the columns will be renamed - and can be defined within the data dictionary later on.
***

In [62]:
# Rename columns
depr = depr.rename(columns={
    'Local Authority District code (2024)': 'lad_code',
    'Index of Multiple Deprivation (IMD) Score': 'imd_score',
    'Income Score (rate)': 'incm_score',
    'Employment Score (rate)': 'empl_score',
    'Education, Skills and Training Score': 'edcn_score',
    'Barriers to Housing and Services Score': 'hous_score'
})

In [63]:
# As data comes from before 2025, we need to update the lad_codes for barnsley and sheffield

# Update old Sheffield LAD code to new Sheffield LAD code
depr.loc[depr['lad_code'] == 'E08000016', 'lad_code'] = 'E08000038'

# Update old Barnsley LAD code to new Barnsley LAD code
depr.loc[depr['lad_code'] == 'E08000019', 'lad_code'] = 'E08000039'

depr[depr['lad_code'].isin(['E08000038', 'E08000039'])]

# Updated succesfully!

,lad_code,imd_score,incm_score,empl_score,edcn_score,hous_score
6955,E08000038,53.446,0.487,0.337,60.621,11.440
6956,E08000038,41.520,0.438,0.256,39.257,10.227
6957,E08000038,14.035,0.124,0.103,12.097,15.412
6958,E08000038,46.044,0.450,0.284,49.686,10.967
6959,E08000038,9.081,0.082,0.093,14.225,18.213
...,...,...,...,...,...,...
32880,E08000039,82.444,0.757,0.468,95.022,18.840
32881,E08000039,6.262,0.066,0.032,1.472,15.861
32882,E08000039,20.432,0.092,0.047,22.716,28.789
32883,E08000039,41.044,0.365,0.241,63.885,14.499


In [64]:
# Merge with lookup table

depr_final = pd.merge(depr, lad_pfr, how='left', on='lad_code')

depr_final.head()

,lad_code,imd_score,incm_score,empl_score,edcn_score,hous_score,lad_name,pfa_code,pfa_name
0,E09000001,8.742,0.013,0.014,0.004,10.950,City of London,E23000034,"London, City of"
1,E09000001,4.722,0.018,0.010,0.169,6.703,City of London,E23000034,"London, City of"
2,E09000001,9.250,0.107,0.064,3.269,9.735,City of London,E23000034,"London, City of"
3,E09000001,19.884,0.211,0.104,17.852,24.623,City of London,E23000034,"London, City of"
4,E09000002,25.307,0.343,0.120,25.442,38.025,Barking and Dagenham,E23000001,Metropolitan Police


In [65]:
# Reorder columns
cols = ['lad_code',
    'lad_name',
    'pfa_code',
    'pfa_name',
    'imd_score',
    'incm_score',
    'empl_score',
    'edcn_score',
    'hous_score']

depr_final = depr_final[cols]

depr_final.head()

,lad_code,lad_name,pfa_code,pfa_name,imd_score,incm_score,empl_score,edcn_score,hous_score
0,E09000001,City of London,E23000034,"London, City of",8.742,0.013,0.014,0.004,10.950
1,E09000001,City of London,E23000034,"London, City of",4.722,0.018,0.010,0.169,6.703
2,E09000001,City of London,E23000034,"London, City of",9.250,0.107,0.064,3.269,9.735
3,E09000001,City of London,E23000034,"London, City of",19.884,0.211,0.104,17.852,24.623
4,E09000002,Barking and Dagenham,E23000001,Metropolitan Police,25.307,0.343,0.120,25.442,38.025


In [66]:
# Check database is clean
depr_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36873 entries, 0 to 36872
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   lad_code    36873 non-null  object 
 1   lad_name    36873 non-null  object 
 2   pfa_code    36873 non-null  object 
 3   pfa_name    36873 non-null  object 
 4   imd_score   36873 non-null  float64
 5   incm_score  36873 non-null  float64
 6   empl_score  36873 non-null  float64
 7   edcn_score  36873 non-null  float64
 8   hous_score  36873 non-null  float64
dtypes: float64(5), object(4)
memory usage: 2.5+ MB


***
All data types are correct
***

In [67]:
# check for null values
depr_final.isnull().sum()

lad_code      0
lad_name      0
pfa_code      0
pfa_name      0
imd_score     0
incm_score    0
empl_score    0
edcn_score    0
hous_score    0
dtype: int64

In [68]:
## Export the code to processed folder as a csv
lookup_table.to_csv('../Data/Processed/lookup-table.csv', index=False)

print(f'File successfully created: {Path('../Data/Processed/lookup-table.csv').exists()}')

File successfully created: True


In [69]:
# check for duplicates
depr_final.duplicated().sum()

np.int64(3118)

***
3118 duplicated values, drop.
***

In [70]:
# Drop duplicate rows
depr_final = depr_final.drop_duplicates()
print(depr_final.duplicated().sum())

0


In [71]:
## Export the code to processed folder as a csv
depr_final.to_csv('../Data/Processed/deprivation-final.csv', index=False)

print(f'deprivation-final.csv File successfully created: {Path('../Data/Processed/deprivation-final.csv').exists()}')

deprivation-final.csv File successfully created: True


#### Crime Severity Weighting

***
This section will create a processed dataset from the crime severity which will find the average weighting for each crime category, to be used in the aggregated dataset in order to aid in visualising where dangerous areas are, rather than where lots of crime happens. It will output in the /Data/Processed folder
***

In [72]:
## Group Table
sev_by_crime_category = sev.groupby(['Crime Category'])

## Create Columns
num_items = sev_by_crime_category['Weight'].count()

mean_weight = sev_by_crime_category['Weight'].mean()

median_weight = sev_by_crime_category['Weight'].median()

min_weight = sev_by_crime_category['Weight'].min()

max_weight = sev_by_crime_category['Weight'].max()

std_deviation_weight = sev_by_crime_category['Weight'].std()

##Formulate Table
crime_category_severity = pd.DataFrame({
    'num_items': num_items,
    'mean_weight': mean_weight,
    'median_weight': median_weight,
    'min_weight': min_weight,
    'max_weight': max_weight,
    'std_deviation_weight': std_deviation_weight
})

## Visulaise Table
display(crime_category_severity)

,num_items,mean_weight,median_weight,min_weight,max_weight,std_deviation_weight
Crime Category,,,,,,
Bicycle Theft,1,16.000000,16.0,16,16,NaN
Burglary,16,703.250000,438.0,117,2127,697.764765
Criminal damage and arson,12,132.000000,19.0,7,837,255.916890
Drugs,5,105.000000,9.0,3,497,219.157478
No Category,8,280.250000,106.0,106,803,322.648305
Other crime,91,162.813187,86.0,4,4392,459.382603
Other theft,8,143.375000,51.5,7,803,268.795694
Possession of weapons,7,367.142857,75.0,55,1365,490.724101
Public order,8,405.500000,261.0,10,1880,615.286461


***
Looking at these stats, taking the median seems to give a better value, as the skew from large and small data is much less.  
Furthermore, by taking the average - given we are working with large datasets - the skew will become more obvious. This is because lower weighted crimes will be committed more often.  
With median: The more common crimes will be weighted slightly higher than they should, the more dangerous crimes will be rated much lower than they should.  
With mean: The more common crimes will be rated much higher than they should, the more dangerous crimes will be rated lower than they should.  
  
**Assumption:** Median is the best average to use for crime severity weighting.
***

In [73]:
## Create Finalised Table
crime_category_severity_final = pd.DataFrame({
    'avg_weight': median_weight,
})

display(crime_category_severity_final) #remove to view table before exporting

## Validation Checks
print(crime_category_severity_final.info())

print(crime_category_severity_final.isnull().sum())

print(crime_category_severity_final.duplicated().sum())


## Output Final Table to csv
crime_category_severity_final.to_csv('../Data/Processed/crime-category-severity-weighting.csv', index=True)

print(f'File successfully created: {Path('../Data/Processed/crime-category-severity-weighting.csv').exists()}')

,avg_weight
Crime Category,
Bicycle Theft,16.0
Burglary,438.0
Criminal damage and arson,19.0
Drugs,9.0
No Category,106.0
Other crime,86.0
Other theft,51.5
Possession of weapons,75.0
Public order,261.0


<class 'pandas.core.frame.DataFrame'>
Index: 14 entries, Bicycle Theft to Violence and sexual offences
Data columns (total 1 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   avg_weight  14 non-null     float64
dtypes: float64(1)
memory usage: 224.0+ bytes
None
avg_weight    0
dtype: int64
1
File successfully created: True


## Aggregation for Reporting

## Export